In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="thivux/phoaudiobook", 
                  repo_type="dataset", local_dir="./phoaudiobook")

In [2]:
files = glob('phoaudiobook/*/*.parquet')
len(files)

805

In [3]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [4]:
# data = loop((files[:1], 0))

In [5]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 25/25 [20:36<00:00, 49.47s/it]


In [6]:
len(data)

1040366

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'phoaudiobook_audio/phoaudiobook-data-train-00040-of-00803_0.mp3',
 'text': 'thì lối ăn mặc và nói năng cũng cần phải cẩn thận. ba. loại bỏ những chuyện thị phi ở đời, vấn đề thị phi được thấy.',
 'speaker': 'phoaudiobook_audio_Huệ_Tâm'}

In [9]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'phoaudiobook')

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  7.22ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 66.5MB / 66.7MB, 6.52MB/s  
Processing Files (1 / 1): 100%|██████████| 66.7MB / 66.7MB, 6.54MB/s  
Processing Files (1 / 1): 100%|██████████| 66.7MB / 66.7MB, 6.67MB/s  
New Data Upload: 100%|██████████| 66.7MB / 66.7MB, 6.67MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:13<00:00, 13.95s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/19bdf5fdcc2cdd68484e202a6f436d53e22d2265', commit_message='Upload dataset', commit_description='', oid='19bdf5fdcc2cdd68484e202a6f436d53e22d2265', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('phoaudiobook-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [11]:
!du -hs phoaudiobook_audio

31G	phoaudiobook_audio


In [14]:
# !zip -rq phoaudiobook_audio.zip phoaudiobook_audio

In [15]:
# !zip -rq phoaudiobook_audio_neucodec.zip phoaudiobook_audio_neucodec

In [18]:
# !hf upload malaysia-ai/Multilingual-TTS phoaudiobook_audio.zip --repo-type=dataset

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS phoaudiobook_audio_neucodec.zip --repo-type=dataset